# n-step TD法

1-step TD は更新が軽い反面、将来情報の取り込みが遅くなります。n-step TD は「何ステップ先まで実報酬を使うか」を調整して、バイアスと分散の折り合いを取る手法です。


## 参考動画
- [対応動画 1](https://www.youtube.com/watch?v=oHm0eNd1yvE)
@[youtube](https://www.youtube.com/watch?v=oHm0eNd1yvE)

## 参考リンク
- https://www.youtube.com/watch?v=oHm0eNd1yvE


## 背景と目的

n を増やすと実報酬の情報量は増えますが、推定分散も増えます。

このトレードオフを理解できると、環境のノイズ特性に合わせて更新長を設計できます。

同じ軌跡に対して n を変え、目標値がどう変化するかを確認します。


## 最初に解きたい疑問

1. `n=1`、`n=3`、`n=T-t` で、何がどう変わるのか。
2. `V_boot` は、なぜ足されるのか。
3. `n_step_return` の `n` は、どこから数えているのか。
4. なぜ `n` を増やすと、ノイズも増えるのか。
5. どんな環境なら、大きい `n` が有利なのか。


## 先に押さえる言葉

- `n-step return`: nステップ先までの実報酬と、その先の推定値を合わせた目標値です。更新に使う未来の長さを調整できます。
- `bootstrapped value`: 実際の報酬の先を、既存の価値推定で埋めた値です。n-step TD では最後に足します。
- `bias`: 推定が系統的にずれることです。短い `n` はこの影響を受けやすいです。
- `variance`: 推定のぶれの大きさです。長い `n` は実報酬を多く使うぶん、ぶれやすくなります。


## 実行前の見取り図

1. `g1` と `g3` の値が、どの程度違うか。
2. `q_after_1` と `q_after_3` が、異なる目標値で更新されているか。
3. `V_boot` を使うことで、将来部分が埋められているか。


## つまずきやすい点

- `n_step_return` が、実報酬と bootstrap をどう混ぜるかの説明が足りない。
- `n=T-t` では終端までの実報酬を使い切るので、bootstrap 項が落ちて Monte Carlo return に一致することを明示する必要があります。


In [ ]:
import math

gamma = 0.9
rewards = [0.2, 0.0, 1.0, -0.1, 0.5]  # R_{t+1}, R_{t+2}, ...
V_boot = 0.7  # V(S_{t+n})


def n_step_return(rews, n, gamma, v_boot):
    n = min(n, len(rews))
    g = 0.0
    for k in range(n):
        g += (gamma ** k) * rews[k]
    if n < len(rews):
        g += (gamma ** n) * v_boot
    return g

for n in [1, 2, 3, 5]:
    print(f'n={n}: G_t^(n)=', round(n_step_return(rewards, n, gamma, V_boot), 6))


## Q更新への適用

\[Q(s_t,a_t) \leftarrow Q(s_t,a_t)+\alpha\{G_t^{(n)}-Q(s_t,a_t)\}\]

`n=1` が通常の TD、`n=T-t` がモンテカルロ更新に対応します。


In [ ]:
alpha = 0.3
q_old = 0.4
g1 = n_step_return(rewards, 1, gamma, V_boot)
g3 = n_step_return(rewards, 3, gamma, V_boot)

q_after_1 = q_old + alpha * (g1 - q_old)
q_after_3 = q_old + alpha * (g3 - q_old)

print('old Q =', q_old)
print('after 1-step update =', round(q_after_1, 6))
print('after 3-step update =', round(q_after_3, 6))


`n` が大きい更新は遠い報酬を早く反映できます。長期依存が強いタスクでは有効ですが、ノイズが大きい環境では不安定化しやすい点に注意します。
